In [16]:
import pandas as pd
import numpy as np
from src.dataset_manager import DatasetManager
from src.training_manager import GGSTrainingManager
from src.test_manager import TestManager
from src.models.negative_binomial import NegativeBinomialPiecewise

In [12]:
m_train, m_test = DatasetManager.split_dataset()

Motores para entrenamiento: 140
Motores para prueba: 60


In [13]:

Training = GGSTrainingManager(
    id_traing = m_train
)

Test = TestManager(
    id_test = m_test
)

In [9]:
param_grid = {
    'model__alpha': [0.1, 0.5, 1.0, 1.5],
    'model__link_type': ['log', 'identity', 'sqrt'], 
    'model__alpha_reg': [0.0, 0.05, 0.1, 0.5],            
    'model__l1_ratio': [0.0, 0.5, 0.75, 1.0],              
    'model__clipping_threshold': [115, 120, 125, 130]              
}

ggs = Training.group_grid_search(
    model = NegativeBinomialPiecewise(),
    param_grid = param_grid, 
    n_folds=5
    )

Starting Grid Search...


GGS en progreso: 100%|██████████| 3840/3840 [5:44:15<00:00,  5.38s/it]   


In [9]:
def obtener_resumen_final(grid_search_object):
    cols_map = {
        'param_model__link_type': 'Link',
        'param_model__alpha' : 'Alpha',
        'param_model__alpha_reg': 'Penalty',
        'param_model__l1_ratio': 'L1_Ratio',
        'param_model__clipping_threshold': 'Threshold',
        'mean_test_C_index': 'C-Index',
        'mean_test_S_score': 'S-Score',
        'mean_test_MAE': 'MAE',
        'mean_test_RMSE': 'RMSE'
    }
    
    df = pd.DataFrame(grid_search_object.cv_results_)
    df = df[list(cols_map.keys())].rename(columns=cols_map)
    
    # Creamos la columna Success basándonos en si el MAE es un número real
    df['Success'] = df['MAE'].apply(lambda x: 0 if np.isnan(x) else 1)
    
    # Rellenamos los NaNs con valores de castigo para que al ordenar 
    # las fallidas queden al final
    df['S-Score'] = df['S-Score'].fillna(999999)
    df['MAE'] = df['MAE'].fillna(999999)
    
    # Ordenar por éxito y luego por S-Score
    df = df.sort_values(by=['Success', 'S-Score'], ascending=[False, True])
    
    return df


In [13]:
res_df = obtener_resumen_final(ggs)
display(res_df.head(15))

,Link,Alpha,Penalty,L1_Ratio,Threshold,C-Index,S-Score,MAE,RMSE,Success
633,log,1.5,0.05,1.00,115,0.853675,7.975524,18.865577,22.674074,1
630,log,1.5,0.05,0.75,115,0.855432,8.826172,20.253591,24.507686,1
294,log,0.5,0.10,0.75,115,0.858197,8.833283,18.710940,22.938103,1
297,log,0.5,0.10,1.00,115,0.858141,8.949517,17.858564,21.855988,1
489,log,1.0,0.10,1.00,115,0.851272,8.959525,20.834251,24.276119,1
438,log,1.0,0.05,0.75,115,0.857810,9.055530,18.709937,22.975160,1
153,log,0.1,0.50,1.00,115,0.849942,9.168318,18.333627,22.233180,1
150,log,0.1,0.50,0.75,115,0.848834,9.182501,19.446565,23.605988,1
645,log,1.5,0.05,1.00,120,0.846329,9.404571,19.792009,23.859046,1
441,log,1.0,0.05,1.00,115,0.856557,9.518486,17.964788,22.007664,1


In [5]:
param_grid_final = {
    'model__link_type': ['log'],           
    'model__alpha': [1.2, 1.5, 1.8],             
    'model__alpha_reg': [0.01, 0.03, 0.05, 0.07], 
    'model__l1_ratio': [0.85, 0.95, 1.0],        
    'model__clipping_threshold': [110, 115, 120] 
}          


ggs_final = Training.group_grid_search(
    model = NegativeBinomialPiecewise(),
    param_grid = param_grid_final, 
    n_folds=5
    )

Starting Grid Search...


GGS en progreso: 100%|██████████| 540/540 [40:13<00:00,  4.47s/it] 


In [10]:
res_df_final = obtener_resumen_final(ggs_final)
display(res_df_final.head(15))

,Link,Alpha,Penalty,L1_Ratio,Threshold,C-Index,S-Score,MAE,RMSE,Success
56,log,1.5,0.05,1.00,110,0.861822,6.802790,17.937398,21.494475,1
55,log,1.5,0.05,0.95,110,0.862304,6.821856,18.187967,21.816501,1
19,log,1.2,0.05,0.95,110,0.863204,6.838107,17.487525,21.221424,1
20,log,1.2,0.05,1.00,110,0.862595,6.878612,17.317380,21.001197,1
18,log,1.2,0.05,0.85,110,0.864160,6.918418,17.872693,21.722117,1
29,log,1.2,0.07,1.00,110,0.860917,7.014278,18.553840,21.989533,1
28,log,1.2,0.07,0.95,110,0.861360,7.060459,18.853380,22.369936,1
54,log,1.5,0.05,0.85,110,0.862633,7.071106,18.770561,22.548270,1
82,log,1.8,0.03,0.95,110,0.864579,7.072122,17.278193,21.046501,1
81,log,1.8,0.03,0.85,110,0.865146,7.103252,17.636720,21.497121,1


In [17]:
param_grid_tune = {
    'model__link_type': 'log',           
    'model__alpha': 1.5,             
    'model__alpha_reg': 0.05, 
    'model__l1_ratio': 1.0,       
    'model__clipping_threshold': 110 
}       

resul_1 = Test.evaluate_best_model(
    param_grid = param_grid_tune,
    model_class = NegativeBinomialPiecewise,
    training_manager = Training,
    only_last = False
)

display(resul_1)


AttributeError: 'GGSTrainingManager' object has no attribute 'get_training_data'